# SynthHeart · 03 · Trust scorecard: fidelity, utility, privacy

Scores the synthetic patients from notebook 02 against the real data, using the bootcamp's evaluation code
(`implementations/tabular_data/evaluation/quality` and `.../privacy`) plus a few SynthHeart additions.

| Section | Question | Metrics |
|---|---|---|
| **Fidelity** | Do synthetic patients look real? | α-precision / β-recall, KS / TVD per column, CI overlap, correlation-matrix gap + per-pair correlation matrices, mutual-information gap, label balance |
| **Utility** | Can they replace real data for a risk model? | **TSTR vs TRTR** XGBoost AUC on the locked test set (headline), F1 difference |
| **Privacy** | Do they leak the training patients? | DCR, median DCR, NNDR, hitting rate, ε-identifiability, exact copies, **membership-inference attack** |
| **Relationships** | Do the medical patterns survive? | Risk curves by BP / age / cholesterol, BP × age table, permutation-importance agreement |

Every privacy metric compares **train** (seen by the generator) against **holdout** (never seen). The **test** split is
only used to score risk models.

**Run** on the AI Factory with the **SynthHeart (.venv)** kernel. It scores `runs/full` by default; set
`SYNTHHEART_RUN_MODE=smoke` to score the smoke run instead. Outputs (hand-off contract in `PLAN.md`):
`results/03_scorecard_<mode>.json` and `runs/<mode>/models/risk_model_synthetic.json`.

## 0 · Settings

In [ ]:
import os
RUN_MODE = os.environ.get("SYNTHHEART_RUN_MODE", "full")   # which notebook-02 run to score
SCALES = ["synth_scale_1", "synth_scale_2"]   # 1x is scored on everything; 2x only in the utility dose test
PRIMARY = SCALES[0]
print(RUN_MODE, PRIMARY)

## 1 · Setup and load

In [ ]:
import json
import sys
from logging import INFO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
from xgboost import XGBClassifier

here = Path.cwd().resolve()
project = next(p / "implementations" / "synthheart" for p in [here, *here.parents]
               if (p / "implementations" / "synthheart" / "synthheart_common.py").exists())
sys.path.insert(0, str(project))
import synthheart_common as sh
sys.path.insert(0, str(sh.REPO))  # so the bootcamp's evaluation helpers import as implementations.tabular_data...

from midst_toolkit.common.logger import TOOLKIT_LOGGER
from midst_toolkit.common.variables import DEVICE
from midst_toolkit.evaluation.privacy import (DistanceToClosestRecordScore, EpsilonIdentifiabilityRisk, HittingRate,
                                              MedianDistanceToClosestRecordScore, NearestNeighborDistanceRatio)
from midst_toolkit.evaluation.privacy.distance_preprocess import preprocess_for_distance_computation
from midst_toolkit.evaluation.privacy.distance_utils import NormType
from midst_toolkit.evaluation.privacy.epsilon_identifiability_risk import EpsilonIdentifiabilityNorm
from midst_toolkit.evaluation.quality import (AlphaPrecision, CorrelationMatrixDifference,
                                              KolmogorovSmirnovAndTotalVariation, MeanConfidenceIntervalOverlap,
                                              MeanF1ScoreDifference, MutualInformationDifference)
from midst_toolkit.evaluation.quality.confidence_interval_overlap import ConfidenceLevel

from implementations.tabular_data.evaluation.display_utils import log_metrics
from implementations.tabular_data.evaluation.preprocessing import (get_numerical_and_categorical_column_names,
                                                                   preprocess_data_for_alpha_precision_eval,
                                                                   remove_label_column_from_other_columns,
                                                                   syntheval_preprocess)

TOOLKIT_LOGGER.setLevel(INFO)
pd.options.mode.chained_assignment = None
print("device:", DEVICE)

train = pd.read_csv(sh.DATA_DIR / f"{sh.TABLE}.csv")
holdout = pd.read_csv(sh.DATA_DIR / f"{sh.TABLE}_holdout.csv")
test = pd.read_csv(sh.DATA_DIR / f"{sh.TABLE}_test.csv")
meta_info = json.loads((sh.DATA_DIR / "meta_info.json").read_text())

RUN_DIR = sh.PROJECT / "runs" / RUN_MODE
synth = {s: pd.read_csv(RUN_DIR / s / sh.TABLE / "_final" / f"{sh.TABLE}_synthetic_filtered.csv")[sh.ALL_COLS]
         for s in SCALES}
S = synth[PRIMARY]
generation = json.loads((sh.RESULTS_DIR / f"02_generation_summary_{RUN_MODE}.json").read_text())
baseline_auc = json.loads((sh.RESULTS_DIR / "baseline_metrics.json").read_text())["trtr_baseline"]["xgboost"]["test_auc"]

for name, df in [("train", train), ("holdout", holdout), ("test", test), *synth.items()]:
    print(f"{name:14s} {len(df):7,d} rows | cardio=1: {df[sh.TARGET].mean():.3f}")

## 2 · Fidelity: do synthetic patients look real?

Same preprocessing as the bootcamp quality notebook: SynthEval encoding (min-max numerics, ordinal categoricals) for
KS/TVD, CI overlap and correlations; one-hot categoricals for α-precision.

In [ ]:
num_cols, cat_cols = get_numerical_and_categorical_column_names(train, meta_info)
se_train, se_synth, se_holdout = syntheval_preprocess(num_cols, cat_cols, train, S, holdout)
fidelity = {}

# α-precision needs equal-size tables (Synthcity), so truncate the larger one, as the bootcamp notebook does.
ap_real, ap_synth = preprocess_data_for_alpha_precision_eval(train, S, meta_info)
n = min(len(ap_real), len(ap_synth))
fidelity["alpha_precision"] = AlphaPrecision(naive_only=False).compute(ap_real.iloc[:n], ap_synth.iloc[:n])
log_metrics("Alpha Precision", fidelity["alpha_precision"])

fidelity["ks_tvd"] = KolmogorovSmirnovAndTotalVariation(
    categorical_columns=cat_cols, numerical_columns=num_cols, significance_level=0.05, permutations=10,
    do_preprocess=False).compute(se_train, se_synth)
log_metrics("Kolmogorov-Smirnov and Total Variation", fidelity["ks_tvd"])

fidelity["ci_overlap"] = MeanConfidenceIntervalOverlap(
    categorical_columns=cat_cols, numerical_columns=num_cols, confidence_level=ConfidenceLevel(95),
    do_preprocess=False).compute(se_train, se_synth)
log_metrics("Mean Confidence Interval Overlap", fidelity["ci_overlap"])

corr_metric = CorrelationMatrixDifference(
    categorical_columns=cat_cols, numerical_columns=num_cols, compute_mixed_correlations=True, do_preprocess=False)
fidelity["correlation"] = corr_metric.compute(se_train, se_synth)
log_metrics("Correlation Matrix Difference", fidelity["correlation"])

fidelity["cardio_rate"] = {"real_train": round(train[sh.TARGET].mean(), 4), "synthetic": round(S[sh.TARGET].mean(), 4)}
print("cardio rate:", fidelity["cardio_rate"])

### Pairwise correlations

The correlation gap above is one number (Frobenius norm of real − synthetic). Here are the matrices behind it:
Pearson for numeric pairs, Cramér's V for categorical pairs, correlation ratio for mixed pairs. Getting the
matrices with `return_mats=True` follows the bootcamp's `multi_table/evaluation/multi_table_metrics.ipynb`.
Mutual information adds a non-linear dependence check on the categorical columns only (the toolkit warns that it
treats numeric columns as strings).

In [ ]:
mats = corr_metric.syntheval_metric.evaluate(mixed_corr=True, return_mats=True)
real_corr, synth_corr = mats["real_cor_mat"].astype(float), mats["synt_cor_mat"].astype(float)
corr_diff = real_corr - synth_corr
upper = corr_diff.where(np.triu(np.ones(corr_diff.shape, dtype=bool), k=1)).stack()  # each pair once
largest = upper.abs().sort_values(ascending=False).head(5).index
fidelity["correlation"]["largest_pair_gaps"] = {f"{a} x {b}": round(float(corr_diff.loc[a, b]), 4) for a, b in largest}
fidelity["correlation_matrices"] = {"real": real_corr.round(4).to_dict(), "synthetic": synth_corr.round(4).to_dict()}
print("largest pair gaps (real - synthetic):", json.dumps(fidelity["correlation"]["largest_pair_gaps"], indent=2))

fidelity["mutual_information"] = MutualInformationDifference(
    categorical_columns=cat_cols, numerical_columns=num_cols, include_numerical_columns=False,
    do_preprocess=False).compute(se_train, se_synth)
log_metrics("Mutual Information Difference (categorical columns)", fidelity["mutual_information"])

diff_lim = max(0.05, float(corr_diff.abs().max().max()))
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for ax, (title, m, lim) in zip(axes, [("real train", real_corr, 1), ("synthetic", synth_corr, 1),
                                      ("real - synthetic", corr_diff, diff_lim)]):
    im = ax.imshow(m.to_numpy(), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax.set_xticks(range(len(m.columns)), m.columns, rotation=90)
    ax.set_yticks(range(len(m.index)), m.index)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

## 3 · Utility: can synthetic patients replace real ones?

**The headline.** The same XGBoost as notebook 01's baseline, trained on synthetic patients only (TSTR) vs on real
patients (TRTR), both scored on the **locked real test set**. Target: TSTR within 0.02 AUC of TRTR. The 2× table is
the dose test (does more synthetic data help?).

In [ ]:
XGB = dict(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
           eval_metric="logloss", random_state=sh.SEED, n_jobs=-1)  # identical to notebook 01's baseline
X_te, y_te = sh.xy(test)

def fit(df):
    X, y = sh.xy(df)
    return XGBClassifier(**XGB).fit(X, y)

def proba(model):
    return model.predict_proba(X_te)[:, 1]

trtr = fit(train)
tstr = {s: fit(df) for s, df in synth.items()}
utility = {
    "trtr_auc": baseline_auc,
    "trtr_auc_refit": round(float(roc_auc_score(y_te, proba(trtr))), 4),  # should match notebook 01
    "tstr_auc": {s: round(float(roc_auc_score(y_te, proba(m))), 4) for s, m in tstr.items()},
    "tstr_accuracy": {s: round(float(accuracy_score(y_te, proba(m) >= 0.5)), 4) for s, m in tstr.items()},
}
utility["auc_gap"] = {s: round(baseline_auc - a, 4) for s, a in utility["tstr_auc"].items()}
print(json.dumps(utility, indent=2))

fig, ax = plt.subplots(figsize=(5, 5))
for name, m in [("real (TRTR)", trtr), *((f"synthetic {s[-1]}x (TSTR)", m) for s, m in tstr.items())]:
    fpr, tpr, _ = roc_curve(y_te, proba(m))
    ax.plot(fpr, tpr, label=f"{name} AUC {roc_auc_score(y_te, proba(m)):.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate")
ax.set_title("Locked real test set"); ax.legend()
plt.show()

In [ ]:
# Bootcamp utility metric: macro-F1 of classifiers trained on real vs synthetic, scored on the holdout.
f1_num, f1_cat = remove_label_column_from_other_columns(sh.TARGET, num_cols, cat_cols)
utility["f1_difference"] = MeanF1ScoreDifference(
    categorical_columns=f1_cat, numerical_columns=f1_num, label_column=sh.TARGET, folds=5, f1_type="macro",
    do_preprocess=False).compute(se_train, se_synth, se_holdout)
log_metrics("F1 Score Difference", utility["f1_difference"])

# The dashboard's risk model: trained only on synthetic patients (1x).
model_path = RUN_DIR / "models" / "risk_model_synthetic.json"
model_path.parent.mkdir(parents=True, exist_ok=True)
tstr[PRIMARY].save_model(model_path)
print("saved", model_path)

## 4 · Privacy: do synthetic patients leak the training patients?

From the bootcamp's `privacy_evaluation_pipeline.ipynb` and `membership_inference_attack.ipynb`. Distance metrics
use one-hot categoricals and range-normalized numerics; hitting rate and ε-identifiability use SynthEval encoding.

This data is coarse (integer heights, BP mostly in steps of 10, many identical profiles), so chance matches are common.
Hitting rate and exact copies are therefore also measured against the **holdout**, a set the generator never saw.
Leakage only shows if synthetic rows match **train** clearly more often than holdout.

In [ ]:
dist_train, dist_synth, dist_holdout = preprocess_for_distance_computation(meta_info, train, S, holdout)
privacy = {}

privacy["dcr"] = DistanceToClosestRecordScore(NormType("l2"), 1000, do_preprocess=False).compute(
    dist_train, dist_synth, dist_holdout)
privacy["dcr"]["ideal"] = round(len(train) / (len(train) + len(holdout)), 4)  # train share of train + holdout
log_metrics("Distance To Closest Record Score", privacy["dcr"])

privacy["median_dcr"] = MedianDistanceToClosestRecordScore(NormType("l2"), 1000, do_preprocess=False).compute(
    dist_train, dist_synth)
log_metrics("Median Distance To Closest Record Score", privacy["median_dcr"])

privacy["nndr"] = NearestNeighborDistanceRatio(NormType("l2"), 1000, do_preprocess=False).compute(
    dist_train, dist_synth, dist_holdout)
log_metrics("NNDR", privacy["nndr"])

hitting = HittingRate(categorical_columns=cat_cols, numerical_columns=num_cols, hitting_threshold=0.03,
                      do_preprocess=False)
privacy["hitting_rate"] = {"train": hitting.compute(se_train, se_synth)["hitting_rate"],
                           "holdout": hitting.compute(se_holdout, se_synth)["hitting_rate"]}
print("hitting rate:", privacy["hitting_rate"])

privacy["epsilon_identifiability"] = EpsilonIdentifiabilityRisk(
    categorical_columns=cat_cols, numerical_columns=num_cols, norm=EpsilonIdentifiabilityNorm("gower"),
    do_preprocess=False).compute(se_train, se_synth, se_holdout)
log_metrics("Epsilon Identifiability Risk", privacy["epsilon_identifiability"])

In [ ]:
# Exact copies: synthetic rows identical to a real row on all 12 columns. The train sample has the holdout's size,
# so the two counts are directly comparable.
def exact_matches(real):
    return int(S.merge(real.drop_duplicates(), on=sh.ALL_COLS).shape[0])

privacy["exact_copies"] = {
    "train": exact_matches(train),
    "train_sample_holdout_size": exact_matches(train.sample(len(holdout), random_state=sh.SEED)),
    "holdout": exact_matches(holdout),
    "synthetic_rows": len(S),
}
print("exact copies:", privacy["exact_copies"])

### Membership-inference attack (black-box DCR attack)

The attacker holds a balanced set of candidates (half train members, half holdout non-members) and the released
synthetic table only. They guess "member" when a candidate sits close to a synthetic row. **AUC 0.5 = coin flip.**
A weak attack failing is not a privacy proof (see the bootcamp notebook), but a high AUC is a clear red flag.

In [ ]:
def min_l2_distance(query: torch.Tensor, reference: torch.Tensor, batch_size: int = 1024) -> np.ndarray:
    # From the bootcamp MIA notebook: distance from each query row to its nearest synthetic row.
    return torch.cat([torch.cdist(query[i:i + batch_size], reference).min(dim=1).values
                      for i in range(0, len(query), batch_size)]).cpu().numpy()

rng = np.random.default_rng(sh.SEED)
n = min(len(dist_train), len(dist_holdout))
members = dist_train.iloc[rng.choice(len(dist_train), n, replace=False)]
non_members = dist_holdout.iloc[rng.choice(len(dist_holdout), n, replace=False)]
labels = np.r_[np.ones(n, dtype=int), np.zeros(n, dtype=int)]
to_tensor = lambda df: torch.tensor(df.to_numpy(), dtype=torch.float32, device=DEVICE)
dcr = min_l2_distance(to_tensor(pd.concat([members, non_members])), to_tensor(dist_synth))
scores = -dcr  # closer to a synthetic row => more likely a member

fpr, tpr, _ = roc_curve(labels, scores)
# Accuracy: threshold picked on one half (Youden's J), applied to the other half, as in the bootcamp notebook.
perm = rng.permutation(len(labels))
cal, held = perm[: len(perm) // 2], perm[len(perm) // 2:]
c_fpr, c_tpr, c_thr = roc_curve(labels[cal], scores[cal])
tau = c_thr[np.argmax(c_tpr - c_fpr)]
privacy["mia"] = {
    "auc": round(float(roc_auc_score(labels, scores)), 4),
    "tpr_at_10pct_fpr": round(float(tpr[fpr <= 0.10].max()), 4),
    "accuracy": round(float(accuracy_score(labels[held], scores[held] >= tau)), 4),
    "mean_dcr_members": round(float(dcr[labels == 1].mean()), 4),
    "mean_dcr_non_members": round(float(dcr[labels == 0].mean()), 4),
    "candidates_per_class": int(n),
}
print(json.dumps(privacy["mia"], indent=2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(dcr[labels == 1], bins=40, alpha=0.6, label="members (train)")
axes[0].hist(dcr[labels == 0], bins=40, alpha=0.6, label="non-members (holdout)")
axes[0].set_xlabel("distance to closest synthetic row"); axes[0].legend()
axes[1].plot(fpr, tpr, label=f"DCR attack (AUC {privacy['mia']['auc']:.3f})")
axes[1].plot([0, 1], [0, 1], "k--", lw=1, label="coin flip")
axes[1].set_xlabel("false positive rate"); axes[1].set_ylabel("true positive rate"); axes[1].legend()
fig.suptitle("Membership-inference attack"); plt.tight_layout(); plt.show()

## 5 · Relationships: do the medical patterns survive?

Cardio rate by blood-pressure band, age band and cholesterol, real train vs synthetic; the BP × age table; and whether
the real-trained and synthetic-trained models rely on the same features (permutation importance on the test set).

In [ ]:
def bands(df):
    return pd.DataFrame({
        "systolic BP": pd.cut(df["ap_hi"], [0, 120, 140, 999], right=False, labels=["<120", "120-139", ">=140"]),
        "age": pd.cut(df["age"], [0, 50, 60, 999], right=False, labels=["<50", "50-59", ">=60"]),
        "cholesterol": df["cholesterol"].map({0: "normal", 1: "above normal", 2: "well above"}),
        sh.TARGET: df[sh.TARGET],
    })

real_b, synth_b = bands(train), bands(S)
rate = lambda b, f: b.groupby(f, observed=True)[sh.TARGET].mean().round(3)
relationships = {"risk_curves": {f: {"real": rate(real_b, f).to_dict(), "synthetic": rate(synth_b, f).to_dict()}
                                 for f in ["systolic BP", "age", "cholesterol"]}}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (f, curves) in zip(axes, relationships["risk_curves"].items()):
    pd.DataFrame(curves).plot.bar(ax=ax, color=["#2a66cc", "#d4492f"], rot=0)
    ax.set_title(f"cardio rate by {f}"); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

grid = lambda b: b.pivot_table(index="age", columns="systolic BP", values=sh.TARGET, aggfunc="mean", observed=True).round(3)
real_grid, synth_grid = grid(real_b), grid(synth_b)
relationships["bp_age_table"] = {"real": real_grid.to_dict(orient="index"), "synthetic": synth_grid.to_dict(orient="index"),
                                 "max_abs_diff": round(float((real_grid - synth_grid).abs().max().max()), 3)}
print("real\n", real_grid, "\n\nsynthetic\n", synth_grid, "\n\nmax |diff|:", relationships["bp_age_table"]["max_abs_diff"])

In [ ]:
importance = pd.DataFrame({
    name: permutation_importance(m, X_te, y_te, scoring="roc_auc", n_repeats=5, random_state=sh.SEED).importances_mean
    for name, m in [("real", trtr), ("synthetic", tstr[PRIMARY])]
}, index=X_te.columns).sort_values("real")
relationships["importance"] = {
    "spearman": round(float(spearmanr(importance["real"], importance["synthetic"]).statistic), 3),
    "top3_real": importance["real"].nlargest(3).index.tolist(),
    "top3_synthetic": importance["synthetic"].nlargest(3).index.tolist(),
    "values": importance.round(4).to_dict(),
}
importance.plot.barh(figsize=(6, 5), color=["#2a66cc", "#d4492f"],
                     title=f"Permutation importance (Spearman {relationships['importance']['spearman']:.2f})")
plt.show()

## 6 · Acceptance and save

Targets from `PLAN.md`. `results/03_scorecard_<mode>.json` feeds the dashboard's scorecard tab.

In [ ]:
def check(value, ok, target):
    return {"value": round(float(value), 4), "target": target, "pass": bool(ok)}

pass_rate = generation["filter"][PRIMARY]["pass_rate"]
alpha = fidelity["alpha_precision"]["delta_precision_alpha_OC"]
sigs = fidelity["ks_tvd"]["num sigs"]
corr = fidelity["correlation"]["corr_mat_diff"]
tstr_auc = utility["tstr_auc"][PRIMARY]
mia_auc = privacy["mia"]["auc"]
dcr_score, dcr_ideal = privacy["dcr"]["dcr_score"], privacy["dcr"]["ideal"]
copies = privacy["exact_copies"]["train"]

acceptance = {
    "rule_pass_rate": check(pass_rate, pass_rate >= 0.98, ">= 0.98"),
    "alpha_precision": check(alpha, alpha >= 0.90, ">= 0.90"),
    "columns_differing": check(sigs, sigs <= 1, "<= 1"),
    "correlation_gap": check(corr, corr <= 0.10, "<= 0.10"),
    "tstr_auc": check(tstr_auc, tstr_auc >= baseline_auc - 0.02, f">= {baseline_auc - 0.02:.3f}"),
    "mia_auc": check(mia_auc, mia_auc <= 0.55, "<= 0.55"),
    "dcr_score": check(dcr_score, abs(dcr_score - dcr_ideal) <= 0.02, f"{dcr_ideal:.2f} +/- 0.02"),
    "exact_copies_of_train": check(copies, copies == 0, "0"),
}
print(pd.DataFrame(acceptance).T.to_string())

scorecard = {
    "run_mode": RUN_MODE,
    "scored": PRIMARY,
    "rows": {"train": len(train), "holdout": len(holdout), "test": len(test), **{s: len(d) for s, d in synth.items()}},
    "acceptance": acceptance,
    "fidelity": fidelity,
    "utility": utility,
    "privacy": privacy,
    "relationships": relationships,
}
out = sh.RESULTS_DIR / f"03_scorecard_{RUN_MODE}.json"
out.write_text(json.dumps(scorecard, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
print("wrote", out)